In [ ]:
import os
os.environ["PYTHONUTF8"] = "1"

In [ ]:
from transformers import BitsAndBytesConfig, AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
import wandb
import torch

MODEL_NAME = "meta-llama/Llama-3.1-8B"
TRAINING_SET = "./data/SW_train_v1.jsonl"
CV_SET = "./data/SW_cv_v1.jsonl"
OUTPUT_DIR = "./output/suicidebot-v1"

In [ ]:
import gc

if 'model' in dir():
    del model
if 'trainer' in dir():
    del trainer

gc.collect()
torch.cuda.empty_cache()

print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"VRAM free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.2f} GB")

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
tokenizer.chat_template = "{% for message in messages %}{% if message['role'] == 'system' %}<|start_header_id|>system<|end_header_id|>\n\n{{ message['content'] }}<|eot_id|>{% elif message['role'] == 'user' %}<|start_header_id|>user<|end_header_id|>\n\n{{ message['content'] }}<|eot_id|>{% elif message['role'] == 'assistant' %}<|start_header_id|>assistant<|end_header_id|>\n\n{{ message['content'] }}<|eot_id|>{% endif %}{% endfor %}"

print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"VRAM free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.2f} GB")

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

model.enable_input_require_grads()
model.config.use_cache = False
if hasattr(model, 'gradient_checkpointing_disable'):
    model.gradient_checkpointing_disable()

In [ ]:
dataset = load_dataset(
    "json",
    data_files={
        "train": TRAINING_SET,
        "cv": CV_SET
    }
)

print(dataset)

In [ ]:
training_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=False,
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    bf16=False,
    fp16=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    report_to="wandb",
    max_seq_length=256,
    completion_only_loss=True,
)

In [ ]:
import wandb

wandb.finish()

wandb.init(
    project="suicidal-llm",
    name="llama3.1-8b-suicidewatch-v1",
    config={
        "model": MODEL_NAME,
        "rank": lora_config.r,
        "lora_alpha": lora_config.lora_alpha,
        "epochs": training_config.num_train_epochs,
        "batch_size": training_config.per_device_train_batch_size,
        "learning_rate": training_config.learning_rate,
    }
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_config,
    train_dataset=dataset["train"],
    eval_dataset=dataset["cv"],
    processing_class=tokenizer,
)

In [ ]:
# Loss diagnostic — run this before trainer.train() to confirm loss is real
dataloader = trainer.get_train_dataloader()
batch = next(iter(dataloader))
batch = {k: v.to("cuda") for k, v in batch.items()}

with torch.no_grad():
    outputs = model(**batch)

print(f"Loss: {outputs.loss.item()}")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"VRAM free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.2f} GB")

In [ ]:
print("Starting training...")
trainer.train()

In [ ]:
# Save adapter weights
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to {OUTPUT_DIR}")